In [ ]:
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from collections import Counter, defaultdict

BASE     = Path(r'd:\Project\Machine_Learning\Fine-Tune SLM for Medical Chatbot\Data\processed')
ICD_PATH = Path(r'd:\Project\Machine_Learning\Fine-Tune SLM for Medical Chatbot\Data\SimpleTabulation-ICD-11-MMS-en.xlsx')
OUT_DIR  = Path(r'd:\Project\Machine_Learning\Fine-Tune SLM for Medical Chatbot\preprocessing')

plt.rcParams.update({
    'figure.facecolor' : '#0f1117',
    'axes.facecolor'   : '#1a1d27',
    'axes.edgecolor'   : '#3a3d4d',
    'axes.labelcolor'  : '#e0e0e0',
    'axes.titlecolor'  : '#ffffff',
    'xtick.color'      : '#a0a0a0',
    'ytick.color'      : '#a0a0a0',
    'text.color'       : '#e0e0e0',
    'grid.color'       : '#2a2d3d',
    'grid.alpha'       : 0.5,
    'font.family'      : 'DejaVu Sans',
})
print('✅ Config loaded')

In [ ]:
# ══════════════════════════════════════════════════════════════
#  1. LOAD DATA
# ══════════════════════════════════════════════════════════════
def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

def extract_text(row, role):
    for msg in row.get('messages', []):
        if msg['role'] == role:
            return msg['content']
    return ''

train    = load_jsonl(BASE / 'train.jsonl')
val      = load_jsonl(BASE / 'val.jsonl')
test     = load_jsonl(BASE / 'test.jsonl')
all_data = train + val + test

split_labels = (['train'] * len(train) +
                ['val']   * len(val)   +
                ['test']  * len(test))

questions = [extract_text(r, 'user')      for r in all_data]
answers   = [extract_text(r, 'assistant') for r in all_data]
texts     = [f"{q} {a}" for q, a in zip(questions, answers)]

q_lens = [len(q) for q in questions]
a_lens = [len(a) for a in answers]

print(f"✅ Loaded — Train:{len(train):,}  Val:{len(val):,}  Test:{len(test):,}  Total:{len(all_data):,}")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  2. BUILD ICD-11 CHAPTER TAXONOMY
#     Baca dari Excel → ekstrak keyword per chapter → group ke
#     16 kategori penyakit yang bermakna secara klinis.
#
#  Catatan arsitektur:
#   • Sumber keyword: (1) ICD-11 Excel titles [EN], (2) CLINICAL_SUPPLEMENTS [EN+ID]
#   • Data Indonesia (~25.8% train) sebelumnya masuk "Lain-lain" karena
#     tidak ada keyword bahasa Indonesia → sudah diperbaiki di CLINICAL_SUPPLEMENTS
#   • min_word_len=5 hanya berlaku saat ekstraksi ICD-11 titles;
#     keyword di CLINICAL_SUPPLEMENTS ditambahkan langsung tanpa filter panjang
#   • Cross-contamination dihapus: keyword muncul di >MAX_SHARED kategori
# ══════════════════════════════════════════════════════════════
icd_df = pd.read_excel(ICD_PATH)
icd_df['ChapterNo_str'] = icd_df['ChapterNo'].astype(str).str.strip().str.zfill(2)

CHAPTER_GROUPS = {
    '01': 'Infeksi & Parasit',
    '02': 'Neoplasma (Kanker)',
    '03': 'Darah & Imunitas',
    '04': 'Darah & Imunitas',
    '05': 'Endokrin & Metabolik',
    '06': 'Mental & Perilaku',
    '07': 'Mental & Perilaku',
    '08': 'Neurologi',
    '09': 'Mata',
    '10': 'Telinga',
    '11': 'Kardiovaskular',
    '12': 'Pernapasan',
    '13': 'Pencernaan',
    '14': 'Kulit',
    '15': 'Muskuloskeletal',
    '16': 'Genitourinary',
    '17': 'Genitourinary',
    '18': 'Kehamilan & Perinatal',
    '19': 'Kehamilan & Perinatal',
    '20': 'Kelainan Perkembangan',
    '21': 'Gejala & Tanda Klinis',   # fallback
    '22': 'Cedera & Darurat',
}

ICD_STOPWORDS = {
    # Preposisi & artikel
    'of', 'the', 'and', 'or', 'in', 'not', 'for', 'with', 'due', 'to',
    'by', 'on', 'at', 'is', 'are', 'as', 'an', 'a', 'into', 'from',
    # Kata kerja umum
    'have', 'been', 'this', 'that', 'which', 'when', 'been',
    # Kata sifat / kata keterangan yang terlalu umum
    'other', 'certain', 'very', 'much', 'more', 'less', 'only', 'just',
    'late', 'early', 'well', 'high', 'high', 'long', 'short', 'full',
    'large', 'small', 'great', 'good', 'same', 'such', 'even', 'both',
    # Klasifikasi ICD generik
    'specified', 'unspecified', 'nos', 'nec', 'type', 'types', 'without',
    'involving', 'part', 'site', 'related', 'associated', 'caused',
    'following', 'during', 'conditions', 'disease', 'diseases', 'disorder',
    'disorders', 'syndrome', 'syndromes', 'sequelae', 'complication',
    'complications', 'form', 'forms', 'nes',
    # Kata sifat severity / stage
    'chronic', 'acute', 'secondary', 'primary', 'multiple', 'single',
    'bilateral', 'unilateral', 'mixed', 'mild', 'moderate', 'severe',
    'grade', 'stage', 'recurrent', 'persistent',
}

def extract_keywords_from_titles(titles, min_word_len=5):
    """Extract unigrams (≥5 char) dan bigrams dari ICD-11 disease titles."""
    keywords = set()
    for title in titles:
        if not isinstance(title, str):
            continue
        cleaned = re.sub(r'\[.*?\]|\(.*?\)', '', title).lower()
        words = re.findall(r"[a-z][a-z\-']+", cleaned)
        words = [w for w in words if len(w) >= min_word_len and w not in ICD_STOPWORDS]
        for w in words:
            keywords.add(w)
        for i in range(len(words) - 1):
            keywords.add(f"{words[i]} {words[i+1]}")
    return keywords

# Bangun keyword set per group dari ICD-11 titles
group_keywords = defaultdict(set)
for ch_no, group_name in CHAPTER_GROUPS.items():
    ch_titles = icd_df[icd_df['ChapterNo_str'] == ch_no]['Title'].dropna().tolist()
    kws = extract_keywords_from_titles(ch_titles)
    group_keywords[group_name].update(kws)

# ── Supplement: klinis Inggris + BAHASA INDONESIA ──────────────────────────
# Dataset indonesia-medical-qna (~681k QA pasien-dokter Alodokter) ditulis
# dalam bahasa Indonesia. Tanpa keyword Indonesia, seluruh data tersebut
# akan jatuh ke "Lain-lain". Kata-kata pendek (<5 karakter) tetap disertakan
# karena tidak melewati filter min_word_len ICD-11.
CLINICAL_SUPPLEMENTS = {
    'Kardiovaskular': {
        # English
        'heart attack', 'chest pain', 'blood pressure', 'hypertension',
        'cardiac arrest', 'palpitation', 'ecg', 'ekg', 'cholesterol',
        'stent', 'bypass', 'coronary artery', 'heart disease', 'heart failure',
        'atrial fibrillation',
        # Indonesia
        'serangan jantung', 'nyeri dada', 'tekanan darah', 'tekanan darah tinggi',
        'jantung berdebar', 'kolesterol tinggi', 'penyakit jantung',
        'gagal jantung', 'aritmia', 'pembuluh darah', 'varises',
        'hipertensi', 'hipotensi', 'darah tinggi', 'darah rendah',
    },
    'Pernapasan': {
        # English
        'shortness of breath', 'wheezing', 'inhaler', 'nebulizer',
        'breathlessness', 'productive cough', 'sputum', 'bronchial asthma',
        'breathing difficulty', 'chronic cough', 'copd', 'sleep apnea',
        # Indonesia
        'sesak napas', 'batuk berdahak', 'batuk kering', 'batuk kronis',
        'batuk darah', 'mengi', 'asma bronkial', 'radang paru',
        'pneumonia', 'bronkitis', 'tuberkulosis', 'tbc', 'flek paru',
        'napas pendek', 'sulit bernapas', 'hidung tersumbat',
        'pilek', 'flu berat', 'batuk pilek',
    },
    'Pencernaan': {
        # English
        'stomachache', 'stomach pain', 'heartburn', 'indigestion', 'bloating',
        'constipation', 'diarrhea', 'loose stool', 'bowel movement',
        'acid reflux', 'gerd', 'irritable bowel', 'abdominal pain',
        # Indonesia
        'sakit perut', 'nyeri perut', 'mual', 'muntah', 'diare', 'mencret',
        'sembelit', 'susah buang air besar', 'maag', 'asam lambung',
        'kembung', 'perut kembung', 'wasir', 'ambeien', 'usus buntu',
        'radang usus', 'tinja berdarah', 'berak darah', 'susah makan',
        'tidak nafsu makan', 'mual muntah', 'nyeri ulu hati',
    },
    'Endokrin & Metabolik': {
        # English
        'blood sugar', 'blood glucose', 'insulin resistance', 'weight gain',
        'obesity', 'body mass index', 'glycemic', 'hba1c', 'thyroid function',
        'diabetes mellitus', 'type 2 diabetes', 'type 1 diabetes',
        'hypoglycemia', 'hyperglycemia', 'metabolic syndrome',
        # Indonesia
        'gula darah', 'kadar gula', 'diabetes melitus', 'kencing manis',
        'tiroid', 'kelenjar tiroid', 'gondok', 'hipotiroid', 'hipertiroid',
        'kegemukan', 'obesitas', 'berat badan berlebih', 'berat badan turun',
        'berat badan naik', 'kolesterol', 'trigliserida', 'asam urat',
        'kurang gizi', 'gizi buruk', 'malnutrisi',
    },
    'Mental & Perilaku': {
        # English
        'stress', 'burnout', 'panic attack', 'phobia', 'hallucination',
        'suicidal', 'self-harm', 'psychotherapy', 'counseling',
        'cognitive behavioral therapy', 'sleep problem', 'insomnia',
        'nightmares', 'trauma', 'mood swings', 'emotional distress',
        'depression', 'anxiety disorder', 'mental health',
        # Indonesia
        'stres', 'depresi', 'cemas', 'kecemasan', 'gangguan jiwa',
        'gangguan mental', 'sulit tidur', 'susah tidur', 'insomnia',
        'mimpi buruk', 'serangan panik', 'fobia', 'trauma psikologis',
        'halusinasi', 'waham', 'skizofrenia', 'bipolar', 'manik',
        'emosi tidak stabil', 'mudah marah', 'perubahan suasana hati',
        'pikiran negatif', 'ingin menyakiti diri', 'bunuh diri',
        'gangguan makan', 'anoreksia', 'bulimia', 'kecanduan',
    },
    'Muskuloskeletal': {
        # English
        'back pain', 'joint pain', 'muscle pain', 'sprain', 'stiffness',
        'swollen joint', 'physiotherapy', 'herniated disc',
        'sciatica', 'osteoporosis', 'bone density', 'muscle cramp',
        # Indonesia
        'nyeri punggung', 'sakit punggung', 'nyeri pinggang', 'sakit pinggang',
        'nyeri sendi', 'sakit sendi', 'nyeri lutut', 'nyeri bahu',
        'nyeri otot', 'kram otot', 'keseleo', 'terkilir', 'patah tulang',
        'tulang rapuh', 'osteoporosis', 'asam urat', 'rematik',
        'radang sendi', 'artritis', 'sendi kaku', 'bengkak sendi',
        'hernia nukleus', 'saraf kejepit', 'skoliosis', 'fisioterapi',
    },
    'Infeksi & Parasit': {
        # English
        'antibiotic', 'antiviral', 'fever', 'chills', 'vaccination',
        'vaccine', 'contagious', 'outbreak', 'covid-19', 'influenza',
        'common cold', 'sore throat', 'runny nose', 'viral infection',
        # Indonesia
        'demam', 'meriang', 'panas tinggi', 'demam tinggi', 'demam berdarah',
        'dbd', 'malaria', 'tipes', 'tifoid', 'disentri',
        'infeksi bakteri', 'infeksi virus', 'infeksi jamur',
        'antibiotik', 'antivirus', 'vaksin', 'imunisasi',
        'flu', 'pilek', 'batuk pilek', 'radang tenggorokan',
        'sakit tenggorokan', 'amandel', 'gondongan', 'cacar',
        'campak', 'hepatitis', 'hiv', 'aids', 'covid',
        'corona', 'ispa', 'infeksi saluran napas',
    },
    'Neoplasma (Kanker)': {
        # English
        'chemotherapy', 'radiation therapy', 'biopsy', 'tumor marker',
        'remission', 'cancer staging', 'oncologist', 'cancer screening',
        'mammogram', 'cancer treatment', 'cancer risk', 'benign tumor',
        # Indonesia
        'kanker', 'tumor', 'benjolan', 'sel kanker', 'stadium kanker',
        'kemoterapi', 'radioterapi', 'operasi tumor', 'biopsi',
        'kanker payudara', 'kanker serviks', 'kanker paru', 'kanker usus',
        'kanker darah', 'leukemia', 'limfoma', 'tumor jinak', 'tumor ganas',
        'metastasis', 'remisi', 'kekambuhan kanker',
    },
    'Genitourinary': {
        # English
        'urinary tract infection', 'kidney stone', 'erectile dysfunction',
        'menstrual cycle', 'menstruation', 'irregular period', 'fertility',
        'contraception', 'sexually transmitted infection', 'vaginal discharge',
        'testicular pain', 'pelvic pain', 'prostate enlargement',
        # Indonesia
        'infeksi saluran kemih', 'batu ginjal', 'batu saluran kemih',
        'gagal ginjal', 'radang ginjal', 'nyeri berkemih', 'susah kencing',
        'sering kencing', 'kencing berdarah', 'kencing nanah',
        'haid', 'menstruasi', 'mens tidak teratur', 'haid tidak teratur',
        'nyeri haid', 'dismenore', 'keputihan', 'infeksi vagina',
        'disfungsi ereksi', 'impotensi', 'infertilitas', 'mandul',
        'program hamil', 'kesuburan', 'kontrasepsi', 'kb',
        'infeksi menular seksual', 'ims', 'sipilis', 'gonore',
        'prostat', 'pembesaran prostat', 'kista ovarium', 'miom',
        'nyeri panggul', 'kista bartholin',
    },
    'Kehamilan & Perinatal': {
        # English
        'pregnant', 'prenatal care', 'antenatal', 'trimester', 'fetal movement',
        'labor pain', 'normal delivery', 'breastfeeding', 'postpartum',
        'morning sickness', 'ultrasound scan', 'newborn care', 'miscarriage',
        'preeclampsia', 'gestational', 'obstetric', 'caesarean',
        # Indonesia
        'hamil', 'kehamilan', 'kandungan', 'usia kehamilan', 'trimester',
        'mual hamil', 'morning sickness', 'gerakan janin', 'detak jantung janin',
        'usg', 'kontraksi', 'melahirkan', 'persalinan', 'bersalin',
        'caesar', 'operasi caesar', 'normal delivery', 'bidan',
        'menyusui', 'asi', 'asi eksklusif', 'bayi baru lahir',
        'neonatus', 'bayi prematur', 'keguguran', 'abortus',
        'preeklamsia', 'plasenta', 'ketuban pecah', 'nifas',
        'pasca melahirkan', 'depresi pasca melahirkan', 'baby blues',
    },
    'Neurologi': {
        # English
        'headache', 'migraine attack', 'seizure', 'stroke symptoms', 'dizziness',
        'vertigo', 'numbness', 'tingling sensation', 'nerve pain', 'tremor',
        'memory loss', 'brain fog', 'neuropathy', 'epilepsy',
        # Indonesia
        'sakit kepala', 'pusing', 'pusing berputar', 'vertigo', 'migrain',
        'kepala berdenyut', 'mati rasa', 'kesemutan', 'kebas', 'tremor',
        'gemetar', 'kejang', 'epilepsi', 'stroke', 'serangan stroke',
        'lumpuh', 'kelumpuhan', 'saraf kejepit', 'nyeri saraf',
        'pikun', 'demensia', 'alzheimer', 'parkinson',
        'pusing tujuh keliling', 'telinga berdenging disertai pusing',
    },
    'Darah & Imunitas': {
        # English
        'anemia', 'blood test', 'hemoglobin level', 'platelet count',
        'white blood cell', 'red blood cell', 'immune deficiency',
        'autoimmune disease', 'allergic reaction', 'hives', 'anaphylaxis',
        # Indonesia
        'anemia', 'kurang darah', 'hemoglobin rendah', 'hb rendah',
        'trombosit rendah', 'trombositopenia', 'leukosit tinggi',
        'sel darah putih', 'sel darah merah', 'golongan darah',
        'transfusi darah', 'alergi', 'reaksi alergi', 'biduran',
        'gatal-gatal', 'ruam alergi', 'anafilaksis', 'imun lemah',
        'daya tahan tubuh', 'autoimun', 'lupus', 'rheumatoid',
    },
    'Kulit': {
        # English
        'skin rash', 'itching', 'eczema flare', 'psoriasis', 'acne breakout',
        'skin irritation', 'dry skin', 'hair loss', 'dandruff',
        'wound healing', 'scar', 'skin infection',
        # Indonesia
        'ruam kulit', 'ruam merah', 'gatal', 'gatal-gatal', 'bentol',
        'eksim', 'dermatitis', 'psoriasis', 'jerawat', 'komedo',
        'kulit kering', 'kulit berminyak', 'rambut rontok', 'kebotakan',
        'ketombe', 'luka', 'bekas luka', 'keloid', 'bisul', 'abses',
        'infeksi kulit', 'jamur kulit', 'panu', 'kadas', 'kudis',
        'cacar air', 'herpes zoster', 'herpes kulit', 'vitiligo',
        'melanoma', 'kanker kulit',
    },
    'Mata': {
        # English
        'eye pain', 'blurry vision', 'red eye', 'eye discharge',
        'vision loss', 'eye drops', 'color blindness', 'eye strain',
        'conjunctivitis', 'glaucoma', 'cataract',
        # Indonesia
        'sakit mata', 'mata merah', 'mata berair', 'mata gatal',
        'penglihatan kabur', 'penglihatan buram', 'mata minus', 'rabun jauh',
        'rabun dekat', 'katarak', 'glaukoma', 'konjungtivitis', 'belekan',
        'mata bengkak', 'kelopak mata bengkak', 'bintitan', 'hordeolum',
        'mata kering', 'silinder', 'mata silau', 'buta warna',
    },
    'Telinga': {
        # English
        'ear pain', 'hearing loss', 'ear discharge', 'ear infection',
        'ringing in ear', 'hearing aid', 'deafness', 'balance problem',
        # Indonesia
        'sakit telinga', 'nyeri telinga', 'telinga berdenging', 'tinnitus',
        'gangguan pendengaran', 'tuli', 'kurang dengar', 'infeksi telinga',
        'telinga berair', 'telinga keluar cairan', 'telinga gatal',
        'kotoran telinga', 'serumen', 'otitis', 'alat bantu dengar',
    },
    'Cedera & Darurat': {
        # English
        'injury', 'fracture', 'broken bone', 'wound care', 'bleeding',
        'first aid', 'emergency room', 'accident', 'poisoning', 'overdose',
        'burn treatment', 'surgery', 'trauma',
        # Indonesia
        'luka', 'luka robek', 'luka bakar', 'luka sayat', 'luka memar',
        'patah tulang', 'retak tulang', 'fraktur', 'dislokasi', 'terkilir',
        'kecelakaan', 'kecelakaan lalu lintas', 'jatuh', 'terpeleset',
        'pertolongan pertama', 'ugd', 'igd', 'gawat darurat',
        'perdarahan', 'pendarahan', 'mimisan', 'keracunan', 'overdosis',
        'operasi', 'pembedahan', 'pasca operasi', 'jahit luka',
        'gigitan binatang', 'sengatan lebah', 'tenggelam',
    },
    'Gejala & Tanda Klinis': {
        # English
        'diagnosis', 'test result', 'lab result', 'clinical finding',
        'physical examination', 'medical history', 'vital signs',
        # Indonesia
        'diagnosa', 'hasil lab', 'hasil tes', 'hasil pemeriksaan',
        'pemeriksaan fisik', 'riwayat penyakit', 'tanda vital',
        'gejala', 'keluhan', 'apa penyebab', 'kenapa bisa',
        'bagaimana cara', 'apakah normal', 'berapa lama', 'kapan sembuh',
    },
    'Kelainan Perkembangan': {
        # English
        'autism spectrum', 'adhd', 'developmental delay', 'congenital anomaly',
        'birth defect', 'down syndrome', 'intellectual disability',
        'learning disability', 'speech delay',
        # Indonesia
        'autisme', 'autis', 'adhd', 'hiperaktif', 'keterlambatan bicara',
        'keterlambatan perkembangan', 'perkembangan terlambat',
        'sindrom down', 'cacat bawaan', 'kelainan bawaan', 'kongenital',
        'cerebral palsy', 'disabilitas intelektual', 'tunagrahita',
        'kesulitan belajar', 'disleksia',
    },
}

for group_name, extras in CLINICAL_SUPPLEMENTS.items():
    group_keywords[group_name].update(extras)

# ── Hapus keyword cross-contamination (muncul di ≥ 4 kategori) ──────
kw_freq = Counter(kw for kws in group_keywords.values() for kw in kws)
MAX_SHARED = 3
for group in group_keywords:
    group_keywords[group] = {kw for kw in group_keywords[group]
                              if kw_freq[kw] <= MAX_SHARED}

# ── Fallback chain: spesifik → Gejala & Tanda → Lain-lain ────────────
FALLBACK_CATS = frozenset({'Gejala & Tanda Klinis', 'Lain-lain'})
SPECIFIC_CATS = [c for c in group_keywords if c not in FALLBACK_CATS]

print(f"✅ Taxonomy built: {len(group_keywords)} disease groups")
print(f"   Keyword EN+ID, cross-contamination dihapus threshold shared ≤ {MAX_SHARED}")
print(f"\n{'Group':<30} {'Keywords':>8}")
print('-' * 42)
for g, kws in sorted(group_keywords.items(), key=lambda x: -len(x[1])):
    flag = ' [fallback]' if g in FALLBACK_CATS else ''
    print(f"  {g:<28} {len(kws):>8,}{flag}")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  TAHAP 3 — TAG SETIAP SAMPLE DENGAN KATEGORI ICD-11
#  Versi teroptimasi:
#    • 1 regex alternation per grup  (bukan N regex terpisah)
#    • finditer() bukan findall()    (iterasi lazy, tidak kumpul list)
#    • Snippets di-precompute 1×     (bukan slice di dalam setiap tag_sample)
#    • Multiprocessing opsional      (aktifkan via USE_MULTIPROCESSING)
#    • Scoring tetap: unigram=1 | bigram=2 | trigram=3
#    • Fallback chain tetap sama     (spesifik → gejala → lain-lain)
# ══════════════════════════════════════════════════════════════

import re
import numpy as np
from functools import partial

# ──────────────────────────────────────────────────────────────
#  KONFIGURASI
# ──────────────────────────────────────────────────────────────
PREVIEW_LEN         = 800    # karakter pertama yang diperiksa per sample
USE_MULTIPROCESSING = False  # set True jika data >50 k dan CPU tersedia
NUM_WORKERS         = 4      # jumlah proses (ignore jika USE_MULTIPROCESSING=False)

# ──────────────────────────────────────────────────────────────
#  KOMPILASI: 1 PATTERN PER GRUP  (key optimisation)
#
#  Struktur compiled_groups[grup] = {
#      'pattern': re.Pattern,   ← 1 alternation regex untuk seluruh grup
#      'weights': dict[str, int] ← kata-kunci → bobot token
#  }
# ──────────────────────────────────────────────────────────────
compiled_specific = {}   # kategori klinis utama (16 kategori)
compiled_fallback = {}   # 'Gejala & Tanda Klinis' + fallback lainnya

for group, kws in group_keywords.items():
    # Urutkan dari terpanjang ke terpendek agar phrase panjang di-match duluan
    sorted_kws = sorted(kws, key=len, reverse=True)

    weights = {kw.lower(): len(kw.split()) for kw in sorted_kws}

    # Satu pola alternation menggantikan N pola terpisah
    pattern = re.compile(
        r'\b(?:' + '|'.join(re.escape(kw) for kw in sorted_kws) + r')\b',
        re.IGNORECASE
    )

    entry = {'pattern': pattern, 'weights': weights}

    if group in FALLBACK_CATS:
        compiled_fallback[group] = entry
    else:
        compiled_specific[group] = entry

# Cache urutan grup agar tidak di-call dict.items() tiap iterasi
_specific_items  = list(compiled_specific.items())
_fallback_gejala = compiled_fallback.get('Gejala & Tanda Klinis')


# ──────────────────────────────────────────────────────────────
#  FUNGSI SCORING — lazy iterator, tidak membangun list
# ──────────────────────────────────────────────────────────────
def weighted_score(entry: dict, snippet: str) -> int:
    """
    Hitung bobot semua keyword yang match di snippet.
    finditer() berhenti segera setelah iterasi habis — tidak
    mengalokasikan list seperti findall().
    """
    score    = 0
    weights  = entry['weights']
    for m in entry['pattern'].finditer(snippet):
        score += weights.get(m.group().lower(), 1)
    return score


# ──────────────────────────────────────────────────────────────
#  FUNGSI TAGGING  (bekerja pada snippet, bukan teks penuh)
# ──────────────────────────────────────────────────────────────
def tag_snippet(snippet: str) -> str:
    """
    Klasifikasikan satu snippet ke kategori ICD-11.
    Fallback chain: spesifik → gejala & tanda → lain-lain
    """
    # Tahap 1: kategori spesifik — ambil yang weighted score tertinggi
    best       = None
    best_score = 0
    for group, entry in _specific_items:
        s = weighted_score(entry, snippet)
        if s > best_score:
            best_score = s
            best       = group

    if best_score > 0:
        return best

    # Tahap 2: fallback 'Gejala & Tanda Klinis'
    if _fallback_gejala and weighted_score(_fallback_gejala, snippet) > 0:
        return 'Gejala & Tanda Klinis'

    # Tahap 3: tidak terklasifikasi
    return 'Lain-lain'


# ──────────────────────────────────────────────────────────────
#  PRECOMPUTE SNIPPETS  (1× slice, bukan N× di dalam loop)
# ──────────────────────────────────────────────────────────────
print(f"🔄 Tagging {len(texts):,} samples...")

snippets = [t[:PREVIEW_LEN] for t in texts]   # O(N) sekali, bukan O(N×K)

# ──────────────────────────────────────────────────────────────
#  JALANKAN TAGGING
# ──────────────────────────────────────────────────────────────
if USE_MULTIPROCESSING:
    from multiprocessing import Pool
    with Pool(processes=NUM_WORKERS) as pool:
        categories = pool.map(tag_snippet, snippets)
else:
    categories = [tag_snippet(s) for s in snippets]

cats_arr  = np.array(categories)
split_arr = np.array(split_labels)

UNIQUE_CATS = sorted(set(categories), key=lambda c: -(cats_arr == c).sum())

# ──────────────────────────────────────────────────────────────
#  RINGKASAN HASIL
# ──────────────────────────────────────────────────────────────
n_total        = len(categories)
n_classified   = sum(1 for c in categories if c != 'Lain-lain')
n_unclassified = n_total - n_classified

print(f"\n✅ Tagging selesai")
print(f"   Terklasifikasi       : {n_classified:,} ({n_classified/n_total*100:.1f}%)")
print(f"   Tidak terklasifikasi : {n_unclassified:,} ({n_unclassified/n_total*100:.1f}%)")

# ──────────────────────────────────────────────────────────────
#  TABEL DISTRIBUSI PER SPLIT
# ──────────────────────────────────────────────────────────────
header = f"\n{'Kategori':<30} {'Total':>8} {'Train':>8} {'Val':>8} {'Test':>8} {'Train%':>7} {'Val%':>6} {'Test%':>6}"
print(header)
print('─' * 85)

for cat in UNIQUE_CATS:
    mask   = cats_arr == cat
    tot    = mask.sum()
    tr     = (mask & (split_arr == 'train')).sum()
    va     = (mask & (split_arr == 'val')).sum()
    te     = (mask & (split_arr == 'test')).sum()
    tr_pct = tr / len(train) * 100
    va_pct = va / len(val)   * 100
    te_pct = te / len(test)  * 100
    print(f"{cat:<30} {tot:>8,} {tr:>8,} {va:>8,} {te:>8,} {tr_pct:>6.1f}% {va_pct:>5.1f}% {te_pct:>5.1f}%")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  4. VISUALISASI — 3×2 PANELS
#  [0, :] Distribusi kategori per split (stacked bar)
#  [1, :] Heatmap konsistensi % per split × kategori
#  [2, 0] Box plot panjang pertanyaan per top-10 kategori
#  [2, 1] Box plot panjang jawaban per top-10 kategori
# ══════════════════════════════════════════════════════════════
TOP_N     = 14   # top-N kategori untuk panel bawah
top_cats  = [c for c in UNIQUE_CATS if c != 'Lain-lain'][:TOP_N]
all_cats_ordered = top_cats + (['Lain-lain'] if 'Lain-lain' in UNIQUE_CATS else [])

SPLIT_COLORS = {'train': '#4e8ef7', 'val': '#f7a24e', 'test': '#4ef7a2'}
SPLIT_ORDER  = ['train', 'val', 'test']

# ── Hitung counts per (cat, split)
count_matrix = {}
pct_matrix   = {}   # % of split that falls into category
split_sizes  = {'train': len(train), 'val': len(val), 'test': len(test)}

for cat in all_cats_ordered:
    count_matrix[cat] = {}
    pct_matrix[cat]   = {}
    for sp in SPLIT_ORDER:
        cnt = ((cats_arr == cat) & (split_arr == sp)).sum()
        count_matrix[cat][sp] = cnt
        pct_matrix[cat][sp]   = cnt / split_sizes[sp] * 100

# ── Build heatmap array — rows=splits, cols=cats
heat_arr = np.array([[pct_matrix[cat][sp] for cat in all_cats_ordered]
                      for sp in SPLIT_ORDER])

# ── Figure
fig = plt.figure(figsize=(26, 20))
fig.patch.set_facecolor('#0f1117')
gs  = gridspec.GridSpec(3, 2, figure=fig,
                        hspace=0.48, wspace=0.30,
                        left=0.07, right=0.97,
                        top=0.93, bottom=0.05)

ax_bar   = fig.add_subplot(gs[0, :])    # Row 0: stacked bar (full width)
ax_heat  = fig.add_subplot(gs[1, :])    # Row 1: heatmap (full width)
ax_qlen  = fig.add_subplot(gs[2, 0])    # Row 2 left: Q-length
ax_alen  = fig.add_subplot(gs[2, 1])    # Row 2 right: A-length

# ──────────────────────────────────────────────────────────
#  PANEL 1 : Distribusi Kategori — Stacked Bar per Split
# ──────────────────────────────────────────────────────────
y_pos    = np.arange(len(all_cats_ordered))
bar_h    = 0.55
bottom   = np.zeros(len(all_cats_ordered))

for sp in SPLIT_ORDER:
    vals = np.array([count_matrix[cat][sp] for cat in all_cats_ordered])
    ax_bar.barh(y_pos, vals, left=bottom, height=bar_h,
                color=SPLIT_COLORS[sp], label=sp.capitalize(), alpha=0.88)
    bottom += vals

# Annotate total on bar end
for i, cat in enumerate(all_cats_ordered):
    total = sum(count_matrix[cat].values())
    pct   = total / n_total * 100
    ax_bar.text(total + n_total * 0.003, i,
                f'{total:,}  ({pct:.1f}%)', va='center', fontsize=8.5,
                color='#cccccc')

ax_bar.set_yticks(y_pos)
ax_bar.set_yticklabels(all_cats_ordered, fontsize=9)
ax_bar.set_xlim(0, n_total * 1.18)
ax_bar.invert_yaxis()
ax_bar.set_xlabel('Jumlah Sampel', fontsize=10)
ax_bar.set_title('Distribusi Konteks Penyakit per Split (ICD-11 Chapter Taxonomy)',
                 fontsize=13, fontweight='bold', pad=10)
ax_bar.legend(loc='lower right', fontsize=9, framealpha=0.2,
              facecolor='#1a1d27', edgecolor='#3a3d4d', labelcolor='#e0e0e0')
ax_bar.grid(axis='x', alpha=0.3)
ax_bar.spines[['top', 'right', 'left', 'bottom']].set_visible(False)

# ──────────────────────────────────────────────────────────
#  PANEL 2 : Heatmap Konsistensi % per Split × Kategori
#  Tiap sel = % split yg jatuh ke kategori itu.
#  Baris yg seragam = distribusi konsisten.
# ──────────────────────────────────────────────────────────
sns.heatmap(
    heat_arr, ax=ax_heat,
    annot=True, fmt='.1f',
    cmap='YlOrRd',
    xticklabels=all_cats_ordered,
    yticklabels=['Train', 'Val', 'Test'],
    linewidths=0.5, linecolor='#0f1117',
    annot_kws={'size': 7.5},
    cbar_kws={'shrink': 0.6, 'label': '% dari split'},
)
ax_heat.set_title(
    'Konsistensi Distribusi Penyakit per Split (% dari ukuran split)\n'
    '— Warna merata = distribusi seimbang antar Train / Val / Test',
    fontsize=11, fontweight='bold', pad=8
)
ax_heat.tick_params(axis='x', labelsize=8, rotation=30)
ax_heat.tick_params(axis='y', labelsize=9, rotation=0)

# ──────────────────────────────────────────────────────────
#  PANEL 3 & 4 : Panjang Q & A per Kategori (Box Plot)
# ──────────────────────────────────────────────────────────
top10 = [c for c in UNIQUE_CATS if c != 'Lain-lain'][:10]

q_data_by_cat = []
a_data_by_cat = []
for cat in top10:
    mask = [c == cat for c in categories]
    q_data_by_cat.append([q_lens[i] for i, m in enumerate(mask) if m])
    a_data_by_cat.append([a_lens[i] for i, m in enumerate(mask) if m])

# Clip ke 95th percentile agar outlier tidak merusak skala
q_cap = np.percentile([x for lst in q_data_by_cat for x in lst], 95)
a_cap = np.percentile([x for lst in a_data_by_cat for x in lst], 95)
q_data_by_cat = [[min(x, q_cap) for x in lst] for lst in q_data_by_cat]
a_data_by_cat = [[min(x, a_cap) for x in lst] for lst in a_data_by_cat]

bp_q = ax_qlen.boxplot(q_data_by_cat, vert=False, patch_artist=True,
                        medianprops=dict(color='#f7a24e', lw=2),
                        flierprops=dict(marker='.', ms=2, alpha=0.2,
                                        markerfacecolor='#666'),
                        whiskerprops=dict(color='#aaa'),
                        capprops=dict(color='#aaa'))
for patch, color in zip(bp_q['boxes'],
                         sns.color_palette('husl', len(top10))):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax_qlen.set_yticks(range(1, len(top10) + 1))
ax_qlen.set_yticklabels(top10, fontsize=8)
ax_qlen.set_xlabel('Panjang Pertanyaan (karakter)', fontsize=9)
ax_qlen.set_title('Distribusi Panjang Pertanyaan per Kategori\n(Top-10, clipped 95th pct)',
                   fontsize=10, fontweight='bold')
ax_qlen.grid(axis='x', alpha=0.3)
ax_qlen.spines[['top', 'right']].set_visible(False)

bp_a = ax_alen.boxplot(a_data_by_cat, vert=False, patch_artist=True,
                        medianprops=dict(color='#f7a24e', lw=2),
                        flierprops=dict(marker='.', ms=2, alpha=0.2,
                                        markerfacecolor='#666'),
                        whiskerprops=dict(color='#aaa'),
                        capprops=dict(color='#aaa'))
for patch, color in zip(bp_a['boxes'],
                         sns.color_palette('husl', len(top10))):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax_alen.set_yticks(range(1, len(top10) + 1))
ax_alen.set_yticklabels(top10, fontsize=8)
ax_alen.set_xlabel('Panjang Jawaban (karakter)', fontsize=9)
ax_alen.set_title('Distribusi Panjang Jawaban per Kategori\n(Top-10, clipped 95th pct)',
                   fontsize=10, fontweight='bold')
ax_alen.grid(axis='x', alpha=0.3)
ax_alen.spines[['top', 'right']].set_visible(False)

# ── Suptitle & save
fig.suptitle(
    f'Persebaran Konteks Penyakit pada Dataset Medical Chatbot  '
    f'(Total: {n_total:,} | Terklasifikasi: {n_classified/n_total*100:.1f}%)',
    fontsize=15, fontweight='bold', color='white', y=0.97
)

out_png = OUT_DIR / 'distribusi_penyakit_icd11.png'
plt.savefig(out_png, dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print(f"\n✅ Visualisasi disimpan → {out_png}")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  5. SIMPAN RINGKASAN KE CSV + LANGUAGE BREAKDOWN
# ══════════════════════════════════════════════════════════════
rows = []
for cat in UNIQUE_CATS:
    mask  = cats_arr == cat
    total = mask.sum()
    tr    = (mask & (split_arr == 'train')).sum()
    va    = (mask & (split_arr == 'val')).sum()
    te    = (mask & (split_arr == 'test')).sum()
    rows.append({
        'category'     : cat,
        'total'        : int(total),
        'pct_total'    : round(total / n_total * 100, 2),
        'train'        : int(tr),
        'val'          : int(va),
        'test'         : int(te),
        'train_pct'    : round(tr / split_sizes['train'] * 100, 2),
        'val_pct'      : round(va / split_sizes['val']   * 100, 2),
        'test_pct'     : round(te / split_sizes['test']  * 100, 2),
        'split_deviation': round(
            max(tr / split_sizes['train'],
                va / split_sizes['val'],
                te / split_sizes['test']) * 100 -
            min(tr / split_sizes['train'],
                va / split_sizes['val'],
                te / split_sizes['test']) * 100, 2
        ),
    })

summary_df = pd.DataFrame(rows).sort_values('total', ascending=False)
out_csv = OUT_DIR / 'distribusi_penyakit_icd11.csv'
summary_df.to_csv(out_csv, index=False)

print(f"✅ Summary disimpan → {out_csv}")
print(f"\n{'='*88}")
print(f"RINGKASAN DISTRIBUSI KATEGORI (split_deviation = selisih max-min % antar split)")
print(f"{'='*88}")
print(summary_df.to_string(index=False))   # fix: .to_string() → print, bukan display()

# Flag kategori yg split-nya tidak seimbang (deviation > 2%)
imbalanced = summary_df[summary_df['split_deviation'] > 2.0]
if len(imbalanced):
    print(f"\n⚠️  Kategori dengan distribusi tidak merata (deviation > 2%):")
    for _, row in imbalanced.iterrows():
        print(f"   {row['category']:<30} → Train:{row['train_pct']:.1f}%  Val:{row['val_pct']:.1f}%  Test:{row['test_pct']:.1f}%  (dev={row['split_deviation']:.1f}%)")
else:
    print(f"\n✅ Semua kategori terdistribusi seimbang antar split (deviation ≤ 2%)")

# ── Language breakdown: estimasi Indonesia vs Inggris per kategori ────────
# Gunakan proxy keyword Indonesia paling umum untuk mendeteksi bahasa
ID_MARKERS = re.compile(
    r'\b(sakit|nyeri|dokter|obat|pasien|penyakit|saya|anda|yang|untuk|dengan|'
    r'adalah|terima kasih|alodokter|halo|hamil|demam|gatal|batuk|pusing|mual)\b',
    re.IGNORECASE
)

lang_rows = []
for cat in UNIQUE_CATS:
    cat_indices = [i for i, c in enumerate(categories) if c == cat]
    n_cat  = len(cat_indices)
    n_id   = sum(1 for i in cat_indices if ID_MARKERS.search(texts[i]))
    n_en   = n_cat - n_id
    lang_rows.append({
        'category': cat,
        'total'   : n_cat,
        'bahasa_id': n_id,
        'english'  : n_en,
        'pct_id'   : round(n_id / n_cat * 100, 1) if n_cat > 0 else 0,
    })

lang_df = pd.DataFrame(lang_rows).sort_values('total', ascending=False)

print(f"\n{'='*65}")
print(f"ESTIMASI DISTRIBUSI BAHASA PER KATEGORI")
print(f"{'='*65}")
print(lang_df.to_string(index=False))

total_id = lang_df['bahasa_id'].sum()
total_en = lang_df['english'].sum()
print(f"\n📊 Keseluruhan dataset: Indonesia ~{total_id:,} ({total_id/n_total*100:.1f}%)  |  "
      f"English ~{total_en:,} ({total_en/n_total*100:.1f}%)")